### LLM as a Judge

In [8]:
import pandas as pd

df_answers = pd.read_csv("rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

#### A->Q->A' evaluation

We'll compare the RAG answer with the original answer from the FAQ. This checks if the RAG pipeline is producing answers that match the ground truth.

In [1]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

First, write the judge instructions. This tells the judge what to compare and how to assign the score.

In [3]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [4]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [5]:
import sys
sys.path.append("..")

In [6]:
from dotenv import load_dotenv
from openai import OpenAI
from src.evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

load_dotenv()
openai_client = OpenAI()

In [10]:
rec = answers[0]
rec

{'question': 'I just found this course — is it still okay to join now, or am I too late?',
 'answer_llm': 'Yes — you can still join now. If you want a certificate, make sure you submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [11]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

In [12]:
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the key meaning of the ground truth: it says it is still okay to join now, and that a certificate requires submitting the project before submissions close. This is semantically equivalent.', score='good')

In [13]:
calc_price(usage)

{'input_cost': 0.00022349999999999998,
 'output_cost': 0.000252,
 'total_cost': 0.0004755}

Now put the same logic into a function:

In [14]:
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [15]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the key point: it is still okay to join now, but certificate eligibility depends on submitting the project before submissions close. This is semantically equivalent to the ground truth.', score='good')

### Running the judge

In [16]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [17]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=5) as pool:
    results = map_progress(pool, answers, judge_record)

  0%|          | 0/720 [00:00<?, ?it/s]

In [18]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [19]:
df_eval = pd.DataFrame(evaluations)

In [20]:
calc_total_price(usages)

0.5171414999999995

Check the results:

In [21]:
good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

Good: 688/720 = 95.56%


Look at the "bad" cases to understand what went wrong:

In [22]:
df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning
3,What do I need to do to be eligible for the ce...,74eb249bbf,bad,"The ground truth says that if you start now, y..."
28,Can I prepare the capstone project before the ...,69d122f12e,bad,The AI answer does not convey the ground truth...
33,"If homework isn’t mandatory, what actually dec...",9f689c185f,bad,The ground truth says the certificate requires...
42,"Is there an upcoming run of the course, and if...",bd31146b0e,bad,The ground truth states that there is an upcom...
57,How do I know when a live session is happening...,d65e05bd7a,bad,The AI answer captures the core points that li...


In [24]:
df_eval.iloc[3]['question']

'What do I need to do to be eligible for the certificate if I start the course now?'

In [25]:
df_eval.iloc[3]['reasoning']

'The ground truth says that if you start now, you can still be eligible for a certificate, but only if you submit your project while submissions are still being accepted. The AI answer adds extra requirements (live cohort, peer reviews) that are not in the ground truth and could be misleading. It does include the key idea that project submission must happen while a live cohort is still accepting submissions, but because it introduces additional, unsupported conditions, it is not semantically equivalent.'

In [26]:
df_eval.to_csv("rag-evaluations-new.csv", index=False)